# HyenaDNA -- Geometric Stability Scaling Experiment

Tests geometric stability of **HyenaDNA**, a long-convolution model based on
the Hyena operator.

## Why HyenaDNA Matters

HyenaDNA uses **implicit long convolutions** (the Hyena operator) instead of
attention or SSM mechanisms:

- **Transformers** (ESM-2, NT): Quadratic attention
- **SSMs** (Caduceus/Mamba): Selective state spaces
- **GPN**: Standard convolutions
- **HyenaDNA**: Implicit long convolutions (continuous, sub-quadratic)

This is architecturally distinct from both Transformers and SSMs, providing
another control for the geometric tax hypothesis.

## Models (5 sizes, 0.5M to 55M)

HyenaDNA comes in sizes from tiny (~0.5M) to large (~55M),
with increasing context windows (1k to 1M). We use seq_length=1000
for fair comparison across all sizes.

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies
#
# HyenaDNA uses einops for the Hyena operator layers.

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

import transformers, torch
print(f"transformers {transformers.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import einops
print(f"einops {einops.__version__}")
print("\nDone!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
PHASE = 'full'

OUTPUT_BASE = './results/hyenadna_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'models': [
            'LongSafari/hyenadna-tiny-1k-seqlen-hf',         # ~0.5M, 1k ctx
            'LongSafari/hyenadna-medium-160k-seqlen-hf',     # ~14M, 160k ctx
        ],
        'batch_size': 16,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'models': [
            'LongSafari/hyenadna-tiny-1k-seqlen-hf',         # ~0.5M, 1k ctx
            'LongSafari/hyenadna-tiny-1k-seqlen-d256-hf',    # ~1.7M, 1k ctx
            'LongSafari/hyenadna-small-32k-seqlen-hf',       # ~4.1M, 32k ctx
            'LongSafari/hyenadna-medium-160k-seqlen-hf',     # ~14M, 160k ctx
            'LongSafari/hyenadna-large-1m-seqlen-hf',        # ~55M, 1M ctx
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

# Large model needs smaller batch
BATCH_OVERRIDES = {
    'hyenadna-large-1m-seqlen-hf': 2,
    'hyenadna-tiny-1k-seqlen-hf': 16,
    'hyenadna-tiny-1k-seqlen-d256-hf': 16,
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: HyenaDNA (Long Convolution / Hyena Operator)")
print(f"Sequences: {config['n_sequences']}")
print(f"Sequence length: {config['seq_length']} nucleotides")
print(f"Models: {len(config['models'])}")

In [ ]:
# Generate Synthetic DNA Sequences

import numpy as np

DNA_BASES = ['A', 'C', 'G', 'T']

def generate_dna_sequences(n_sequences, seq_length, seed=320):
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(DNA_BASES, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f"Generated {len(sequences)} DNA sequences (length={seq_length})")
    print(f"Sample: {sequences[0][:50]}...")
    return sequences

sequences = generate_dna_sequences(config['n_sequences'], config['seq_length'])

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_dna(sequence, mutation_rate, rng):
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results


suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])

test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:5], pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# HyenaDNA Model Wrapper
#
# HyenaDNA uses character-level tokenization and the Hyena operator
# (implicit long convolutions). All HF models end with -hf suffix.

import torch
import gc
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_hyenadna_embedding_fn(model_name, batch_size=8):
    """Load HyenaDNA and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    short = model_name.split('/')[-1]
    effective_batch = BATCH_OVERRIDES.get(short, batch_size)
    if effective_batch != batch_size:
        print(f"  Batch size override: {batch_size} -> {effective_batch}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # HyenaDNA models may need AutoModel or AutoModelForCausalLM
    try:
        model = AutoModel.from_pretrained(
            model_name, trust_remote_code=True,
        ).to(device).eval()
    except Exception as e:
        print(f"  AutoModel failed ({e}), trying AutoModelForCausalLM...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, trust_remote_code=True,
        ).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    # HyenaDNA uses character-level tokenization; cap at seq_length
    max_length = min(config['seq_length'] + 2, 1024)  # +2 for special tokens
    print(f"  Max token length: {max_length}")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + effective_batch - 1) // effective_batch

        for i in range(0, len(sequences), effective_batch):
            batch = sequences[i:i + effective_batch]
            batch_idx = i // effective_batch + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            try:
                outputs = model(**tokens, output_hidden_states=True)
                hidden = outputs.hidden_states[-1]
            except Exception:
                outputs = model(**tokens)
                if hasattr(outputs, 'hidden_states') and outputs.hidden_states:
                    hidden = outputs.hidden_states[-1]
                elif hasattr(outputs, 'last_hidden_state'):
                    hidden = outputs.last_hidden_state
                else:
                    hidden = outputs[0]

            # Mean pooling (with attention mask if available)
            if 'attention_mask' in tokens:
                mask = tokens['attention_mask'].unsqueeze(-1)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            else:
                pooled = hidden.mean(dim=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("HyenaDNA wrapper ready")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'HYENADNA (LONG CONVOLUTION) SCALING -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    embed_fn, model_obj, tokenizer_obj, n_params_m = make_hyenadna_embedding_fn(
        model_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_').replace('-', '_')
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached clean embeddings: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/hyenadna_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualization

import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12})

summaries = [r.summary() for r in reports]
sizes = model_param_counts

metrics = {
    'mean_composite_stability': ('Composite Stability', 'tab:olive'),
    'mean_rdm_similarity_score': ('RDM Similarity', 'tab:green'),
    'mean_perturbation_stability_score': ('Perturbation Stability', 'tab:purple'),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (metric_key, (label, color)) in enumerate(metrics.items()):
    ax = axes[idx]
    values = [s[metric_key] for s in summaries]
    std_key = metric_key.replace('mean_', 'std_')
    errors = [s.get(std_key, 0) for s in summaries]

    ax.errorbar(sizes, values, yerr=errors, fmt='o-', color=color,
                linewidth=2, markersize=10, capsize=5)
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel(label)
    ax.set_title(label, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if max(sizes) / max(min(sizes), 0.01) > 10:
        ax.set_xscale('log')

fig.suptitle(
    'HyenaDNA (Long Convolution): Geometric Stability vs. Model Size',
    fontsize=14, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/hyenadna_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {RESULTS_DIR}/hyenadna_scaling_{PHASE}.png')

In [ ]:
# Cross-Experiment Comparison

import matplotlib.pyplot as plt
import pandas as pd
import glob

fig, ax = plt.subplots(figsize=(10, 6))

# HyenaDNA (this experiment)
hyena_agg = df_detailed.groupby(['model', 'size_M'])['composite'].mean().reset_index()
ax.plot(hyena_agg['size_M'], hyena_agg['composite'], 'v-',
        color='tab:olive', linewidth=2, markersize=10,
        label='HyenaDNA (Long Conv)', zorder=3)

# Other architectures
for label, pattern, color, marker in [
    ('NT (Transformer)', '**/nt_scaling_*_detailed.csv', 'tab:orange', 's-'),
    ('ESM-2 (Transformer)', '**/esm2_scaling_*_detailed.csv', 'tab:red', 'o-'),
    ('Caduceus (SSM)', '**/caduceus_scaling_*_detailed.csv', 'tab:blue', 'D-'),
    ('GPN (Conv)', '**/gpn_scaling_*_detailed.csv', 'tab:cyan', '^-'),
    ('DNABERT-2', '**/dnabert2_scaling_*_detailed.csv', 'tab:green', 'P'),
]:
    files = glob.glob(f'{RESULTS_DIR}/../../../{pattern}', recursive=True)
    if files:
        df = pd.read_csv(files[0])
        agg = df.groupby('size_M')['composite'].mean().reset_index()
        ax.plot(agg['size_M'], agg['composite'], marker,
                color=color, linewidth=2, markersize=10, label=label)

ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('Full Architecture Comparison: The Geometric Tax', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/hyenadna_crossexp_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Slope Analysis ---')
if len(hyena_agg) >= 2:
    slope = hyena_agg['composite'].iloc[-1] - hyena_agg['composite'].iloc[0]
    print(f'HyenaDNA composite change: {slope:+.4f} (first to last)')